
# Module 0 - Setup (fixed)
Lands public datasets, fetches a few real CISA advisory PDFs, generates synthetic ones, populates volumes.

In [ ]:
%run ./_config


In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {BASE}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {BASE}.raw_advisories")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {BASE}.raw_stigs")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {BASE}.ka_corpus")


In [ ]:
%pip install -q reportlab requests
dbutils.library.restartPython()

## CISA KEV catalog

In [ ]:
import requests
import json
from pyspark.sql import functions as F

KEV_URL = "https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.json"
raw = requests.get(KEV_URL, timeout=30).json()
vulns = raw["vulnerabilities"]
print(f"Fetched {len(vulns)} KEV entries (catalog version {raw['catalogVersion']})")

df = spark.createDataFrame(vulns)
df = df.withColumn("dateAdded", F.to_date("dateAdded")).withColumn("dueDate", F.to_date("dueDate"))
df.write.mode("overwrite").saveAsTable(f"{BASE}.kev_catalog")
df.limit(5).display()


## CVEs - real data from NVD 2.0 REST API
The legacy NVD JSON feeds (1.1) are 403 for unauth bulk access. The 2.0 REST API
works without an API key (rate-limited to 5 req / 30s). We page through the
last several months of CVEs and merge in any KEV entries we don't already have.

In [ ]:
from datetime import datetime, timedelta, timezone
import time

NVD_BASE = "https://services.nvd.nist.gov/rest/json/cves/2.0"

def fetch_nvd_window(start_iso, end_iso, page_size=2000):
    """Generator yielding NVD CVE records for a publication window."""
    start_index = 0
    while True:
        url = (f"{NVD_BASE}?pubStartDate={start_iso}&pubEndDate={end_iso}"
               f"&resultsPerPage={page_size}&startIndex={start_index}")
        for attempt in range(3):
            try:
                r = requests.get(url, timeout=60, headers={"User-Agent": "DISA-Workshop"})
                if r.status_code == 200:
                    break
                print(f"   NVD HTTP {r.status_code}, sleeping 8s and retrying ({attempt+1}/3)")
                time.sleep(8)
            except Exception as e:
                print(f"   NVD error {e}, retry")
                time.sleep(8)
        else:
            print(f"   NVD giving up on {start_iso}..{end_iso}")
            return
        data = r.json()
        for v in data.get("vulnerabilities", []):
            yield v["cve"]
        total = data.get("totalResults", 0)
        start_index += data.get("resultsPerPage", page_size)
        if start_index >= total or not data.get("vulnerabilities"):
            return
        time.sleep(7)  # rate-limit cushion

# Pull last 6 months in 30-day windows
end = datetime.now(timezone.utc).replace(microsecond=0)
all_cves = {}
for delta in range(0, 180, 30):
    win_end = end - timedelta(days=delta)
    win_start = win_end - timedelta(days=30)
    s = win_start.strftime("%Y-%m-%dT%H:%M:%S.000")
    e = win_end.strftime("%Y-%m-%dT%H:%M:%S.000")
    print(f"Fetching NVD window {s} to {e} ...")
    for cve in fetch_nvd_window(s, e):
        cve_id = cve.get("id")
        if not cve_id or cve_id in all_cves:
            continue
        descs = cve.get("descriptions", [])
        description = next((d.get("value","") for d in descs if d.get("lang") == "en"), "")
        cvss = None
        for key in ("cvssMetricV31", "cvssMetricV30", "cvssMetricV2"):
            metrics = cve.get("metrics", {}).get(key, [])
            if metrics:
                cvss = metrics[0].get("cvssData", {})
                break
        all_cves[cve_id] = {
            "cve_id": cve_id,
            "description": (description or "")[:2000],
            "published_date": (cve.get("published") or "2024-01-01")[:10],
            "cvss_score": float(cvss.get("baseScore")) if cvss and cvss.get("baseScore") is not None else None,
            "cvss_severity": (cvss.get("baseSeverity") if cvss else None) or None,
            "attack_vector": (cvss.get("attackVector") if cvss else None) or None,
        }
    time.sleep(7)

print(f"NVD pulled {len(all_cves)} unique CVEs from last 6 months")

# Merge in any KEV entries we missed (so KEV joins always work)
kev_rows = spark.table(f"{BASE}.kev_catalog").collect()
added_from_kev = 0
for k in kev_rows:
    if not k.cveID or k.cveID in all_cves:
        continue
    all_cves[k.cveID] = {
        "cve_id": k.cveID,
        "description": (k.shortDescription or k.vulnerabilityName or "")[:2000],
        "published_date": str(k.dateAdded) if k.dateAdded else "2024-01-01",
        "cvss_score": 8.5,
        "cvss_severity": "HIGH",
        "attack_vector": "NETWORK",
    }
    added_from_kev += 1
print(f"Added {added_from_kev} additional CVEs from KEV catalog (total {len(all_cves)})")

rows = list(all_cves.values())
df = spark.createDataFrame(rows).withColumn("published_date", F.to_date("published_date"))
df.write.mode("overwrite").saveAsTable(f"{BASE}.cves")
print(f"Wrote {df.count()} CVEs to {BASE}.cves")


## MITRE ATT&CK

In [ ]:
ATTACK_URL = "https://raw.githubusercontent.com/mitre/cti/master/enterprise-attack/enterprise-attack.json"
bundle = requests.get(ATTACK_URL, timeout=120).json()

techniques, groups = [], []
for obj in bundle["objects"]:
    if obj["type"] == "attack-pattern":
        ext_refs = obj.get("external_references", [])
        attack_id = next((r["external_id"] for r in ext_refs if r.get("source_name") == "mitre-attack"), None)
        techniques.append({
            "technique_id": attack_id,
            "name": obj.get("name"),
            "description": obj.get("description", "")[:2000],
            "is_subtechnique": obj.get("x_mitre_is_subtechnique", False),
            "platforms": obj.get("x_mitre_platforms", []),
        })
    elif obj["type"] == "intrusion-set":
        ext_refs = obj.get("external_references", [])
        group_id = next((r["external_id"] for r in ext_refs if r.get("source_name") == "mitre-attack"), None)
        groups.append({
            "group_id": group_id,
            "name": obj.get("name"),
            "aliases": obj.get("aliases", []),
            "description": obj.get("description", "")[:2000],
        })

print(f"Loaded {len(techniques)} techniques, {len(groups)} threat groups")
spark.createDataFrame(techniques).write.mode("overwrite").saveAsTable(f"{BASE}.attack_techniques")
spark.createDataFrame(groups).write.mode("overwrite").saveAsTable(f"{BASE}.attack_groups")


## Synthetic affected assets

In [ ]:
import random
from datetime import date

random.seed(42)

VENDORS = ["Microsoft", "Cisco", "Adobe", "VMware", "Apache", "Fortinet", "Citrix"]
PRODUCTS = {
    "Microsoft": ["Windows Server 2019", "Exchange Server", "Office 365"],
    "Cisco": ["IOS XE", "ASA", "Catalyst Switch"],
    "Adobe": ["Acrobat Reader", "ColdFusion"],
    "VMware": ["vCenter", "ESXi"],
    "Apache": ["HTTP Server", "Log4j", "Struts"],
    "Fortinet": ["FortiOS", "FortiGate"],
    "Citrix": ["NetScaler ADC", "XenApp"],
}
ENVS = ["NIPRNet", "SIPRNet", "JWICS"]

assets = []
for i in range(500):
    vendor = random.choice(VENDORS)
    product = random.choice(PRODUCTS[vendor])
    assets.append({
        "asset_id": f"AST-{i:05d}",
        "hostname": f"host-{i:04d}.disa.mil",
        "environment": random.choice(ENVS),
        "vendor": vendor,
        "product": product,
        "os_family": "Windows" if vendor == "Microsoft" else "Linux" if vendor in ("Apache",) else "Network",
        "last_patched": str(date(2026, random.randint(1, 4), random.randint(1, 28))),
        "owner_org": random.choice(["DISA-J3", "DISA-J6", "USCYBERCOM", "JFHQ-DODIN"]),
    })

df = spark.createDataFrame(assets).withColumn("last_patched", F.to_date("last_patched"))
df.write.mode("overwrite").saveAsTable(f"{BASE}.affected_assets")
print(f"Created {df.count()} synthetic assets")


## Advisory PDFs - scrape live CISA RSS, then fall back to synthetic
We scrape the CISA cybersecurity advisories RSS, follow each alert page, find
linked PDFs, and download them. Synthetic PDFs are added at the end so the
corpus always covers the major vendor/product mix used in Modules 3-5.

In [ ]:
import re
from urllib.parse import urljoin

ADVISORY_VOL = VOL_ADVISORIES

# Clean any stale files from prior runs so the corpus is consistent
import os
for fn in os.listdir(ADVISORY_VOL):
    try:
        os.remove(f"{ADVISORY_VOL}/{fn}")
    except Exception:
        pass

CISA_FEEDS = [
    "https://www.cisa.gov/cybersecurity-advisories/all.xml",
    "https://www.cisa.gov/uscert/ncas/alerts.xml",
]

# Probe one feed first; if cisa.gov egress is blocked we skip the whole scrape.
try:
    probe = requests.get(CISA_FEEDS[0], timeout=10, headers={"User-Agent": "Mozilla/5.0"})
    cisa_reachable = probe.status_code == 200 and "<link>" in probe.text
except Exception:
    cisa_reachable = False
print(f"CISA reachable from workspace? {cisa_reachable}")

alert_links = set()
if cisa_reachable:
    for feed in CISA_FEEDS:
        try:
            rss = requests.get(feed, timeout=20, headers={"User-Agent": "Mozilla/5.0"}).text
            for m in re.finditer(r"<link>(https://www\.cisa\.gov/news-events/[^<]+)</link>", rss):
                alert_links.add(m.group(1))
        except Exception as e:
            print(f"WARN: feed {feed} failed: {e}")
    print(f"Found {len(alert_links)} alert links across feeds")

pdf_urls = set()
if cisa_reachable:
    for link in list(alert_links)[:40]:
        try:
            html = requests.get(link, timeout=15, headers={"User-Agent": "Mozilla/5.0"}).text
            for m in re.finditer(r'href="(/sites/default/files/[^"]+\.pdf)"', html):
                pdf_urls.add("https://www.cisa.gov" + m.group(1))
        except Exception:
            pass
print(f"Discovered {len(pdf_urls)} unique CISA PDF URLs")

real_count = 0
for url in list(pdf_urls)[:25]:  # cap at 25 to keep ai_parse_document runs fast
    fname = url.rsplit("/", 1)[-1]
    try:
        r = requests.get(url, timeout=60, headers={"User-Agent": "DISA-Workshop"})
        r.raise_for_status()
        if r.headers.get("content-type", "").startswith("application/pdf") or r.content[:4] == b"%PDF":
            with open(f"{ADVISORY_VOL}/{fname}", "wb") as f:
                f.write(r.content)
            print(f"  + {fname} ({len(r.content)} bytes)")
            real_count += 1
        else:
            print(f"  - {fname}: not a PDF (content-type={r.headers.get('content-type')})")
    except Exception as e:
        print(f"  - {url} failed: {e}")

print(f"\nFetched {real_count} real CISA advisory PDFs")


In [ ]:
# Synthetic CISA-style advisory PDFs - only generate if scraping returned <5 real PDFs
# (keeps the demo functional even when egress to cisa.gov is blocked).
if real_count >= 5:
    print(f"Have {real_count} real PDFs; skipping synthetic generation.")
    SKIP_SYNTHETIC = True
else:
    SKIP_SYNTHETIC = False
    print(f"Only {real_count} real PDFs; generating synthetic fallback.")

from reportlab.lib.pagesizes import LETTER
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.units import inch

SYNTH_ADVISORIES = [
    {
        "id": "aa25-001a",
        "title": "Cisco IOS XE Web UI Privilege Escalation Exploited in the Wild",
        "vendor": "Cisco",
        "product": "IOS XE",
        "cves": ["CVE-2023-20198", "CVE-2023-20273"],
        "summary": "CISA and partner agencies have observed active exploitation of Cisco IOS XE Web UI vulnerabilities allowing unauthenticated remote attackers to gain privilege level 15 access and persist via implants.",
        "mitigations": [
            "Disable the HTTP/HTTPS server feature on internet-facing devices (no ip http server / no ip http secure-server).",
            "Apply Cisco IOS XE software fixes per Cisco PSIRT advisory.",
            "Hunt for unauthorized accounts and unexpected privileged users.",
        ],
    },
    {
        "id": "aa25-002a",
        "title": "Microsoft Exchange Server ProxyShell-class Vulnerabilities",
        "vendor": "Microsoft",
        "product": "Exchange Server",
        "cves": ["CVE-2024-21410", "CVE-2024-26198"],
        "summary": "Threat actors continue to target on-premises Microsoft Exchange Servers with privilege escalation and remote code execution exploits, deploying webshells and exfiltrating mail.",
        "mitigations": [
            "Apply the latest Exchange Server cumulative updates and security updates immediately.",
            "Enable Extended Protection for Authentication.",
            "Hunt for ASPX webshells in OWA virtual directories.",
        ],
    },
    {
        "id": "aa25-003a",
        "title": "Apache Log4j Resurgent Exploitation Targeting DoD Systems",
        "vendor": "Apache",
        "product": "Log4j",
        "cves": ["CVE-2021-44228", "CVE-2021-45046"],
        "summary": "Despite four years of remediation, opportunistic actors continue to scan and exploit unpatched Apache Log4j installations across DoD networks. Recent campaigns drop Cobalt Strike beacons via JNDI lookups.",
        "mitigations": [
            "Upgrade Log4j to 2.17.1 or later.",
            "Block outbound LDAP/RMI from server VLANs.",
            "Inspect application logs for ${jndi:} patterns.",
        ],
    },
    {
        "id": "aa25-004a",
        "title": "Fortinet FortiOS SSL-VPN Credential Theft Campaign",
        "vendor": "Fortinet",
        "product": "FortiOS",
        "cves": ["CVE-2022-42475", "CVE-2024-21762"],
        "summary": "Iranian-aligned actors are abusing FortiOS SSL-VPN heap overflow and out-of-bounds write vulnerabilities to obtain initial access on federal networks, exfiltrating credentials and pivoting to Active Directory.",
        "mitigations": [
            "Patch FortiOS to 7.4.3 or later.",
            "Disable SSL-VPN if not required.",
            "Force credential rotation for any account that authenticated through SSL-VPN.",
        ],
    },
    {
        "id": "aa25-005a",
        "title": "VMware vCenter Server Authentication Bypass",
        "vendor": "VMware",
        "product": "vCenter",
        "cves": ["CVE-2023-34048", "CVE-2024-22252"],
        "summary": "An authentication bypass in vCenter Server's DCERPC implementation allows remote unauthenticated code execution. Multiple ransomware groups are weaponizing the chain in DoD-adjacent supply-chain targets.",
        "mitigations": [
            "Apply VMware patches to all vCenter 7.0 and 8.0 instances.",
            "Restrict vCenter management plane to dedicated admin networks.",
            "Audit vpxd.log for anomalous DCERPC calls.",
        ],
    },
]

if not SKIP_SYNTHETIC:
    styles = getSampleStyleSheet()
    for adv in SYNTH_ADVISORIES:
        fname = f"{ADVISORY_VOL}/{adv['id']}-{adv['vendor'].lower()}-{adv['product'].lower().replace(' ','-')}.pdf"
        doc = SimpleDocTemplate(fname, pagesize=LETTER, leftMargin=0.75*inch, rightMargin=0.75*inch)
        story = []
        story.append(Paragraph(f"<b>CISA Cybersecurity Advisory {adv['id'].upper()}</b>", styles["Heading1"]))
        story.append(Paragraph(adv["title"], styles["Heading2"]))
        story.append(Spacer(1, 12))
        story.append(Paragraph("<b>Affected vendor:</b> " + adv["vendor"], styles["BodyText"]))
        story.append(Paragraph("<b>Affected product:</b> " + adv["product"], styles["BodyText"]))
        story.append(Paragraph("<b>CVEs:</b> " + ", ".join(adv["cves"]), styles["BodyText"]))
        story.append(Spacer(1, 12))
        story.append(Paragraph("<b>Summary</b>", styles["Heading3"]))
        story.append(Paragraph(adv["summary"], styles["BodyText"]))
        story.append(Spacer(1, 12))
        story.append(Paragraph("<b>Recommended Mitigations</b>", styles["Heading3"]))
        for m in adv["mitigations"]:
            story.append(Paragraph("- " + m, styles["BodyText"]))
        doc.build(story)
        print(f"Generated synthetic PDF: {fname}")

## STIG corpus - real DoD XCCDF ZIPs from public.cyber.mil + synthetic fallback

In [ ]:
STIG_VOL = VOL_STIGS

# Clean stale files
import os
for fn in os.listdir(STIG_VOL):
    try:
        os.remove(f"{STIG_VOL}/{fn}")
    except Exception:
        pass

REAL_STIG_ZIPS = [
    "https://dl.dod.cyber.mil/wp-content/uploads/stigs/zip/U_RHEL_8_V2R3_STIG.zip",
    "https://dl.dod.cyber.mil/wp-content/uploads/stigs/zip/U_RHEL_9_V2R5_STIG.zip",
    "https://dl.dod.cyber.mil/wp-content/uploads/stigs/zip/U_MS_Windows_Server_2022_V2R5_STIG.zip",
    "https://dl.dod.cyber.mil/wp-content/uploads/stigs/zip/U_MS_Windows_11_V2R5_STIG.zip",
]

import zipfile, io, xml.etree.ElementTree as ET, re

def xccdf_to_text(xccdf_bytes, source_name):
    """Parse XCCDF XML and emit one text snippet per Rule. Strips all XML namespaces."""
    snippets = []
    try:
        text = xccdf_bytes.decode("utf-8", errors="ignore")
        text = re.sub(r'<\?xml-stylesheet[^?]*\?>', '', text)
        text = re.sub(r'xmlns(?::\w+)?="[^"]+"', "", text)
        # strip namespace prefixes from element tags
        text = re.sub(r'<\s*(/?)\s*[a-zA-Z][\w\-]*:', r'<\1', text)
        # strip namespace prefixes from attribute names
        text = re.sub(r'\s([a-zA-Z][\w\-]*):([a-zA-Z])', r' \2', text)
        root = ET.fromstring(text)
    except Exception as e:
        print(f"   parse err: {e}")
        return snippets
    # Find all Rule elements
    for rule in root.iter("Rule"):
        rid = rule.attrib.get("id", "")
        sev = rule.attrib.get("severity", "")
        title_el = rule.find("title")
        title = (title_el.text or "").strip() if title_el is not None else ""
        # version (V-#####)
        ver_el = rule.find("version")
        vid = (ver_el.text or "").strip() if ver_el is not None else ""
        # description (free text, may contain CDATA-ish structure)
        desc_el = rule.find("description")
        desc = (desc_el.text or "").strip() if desc_el is not None else ""
        # check
        check_el = rule.find(".//check-content")
        check = (check_el.text or "").strip() if check_el is not None else ""
        # fix
        fix_el = rule.find("fixtext")
        fix = (fix_el.text or "").strip() if fix_el is not None else ""
        if not (vid or rid):
            continue
        sev_label = {"high": "CAT I", "medium": "CAT II", "low": "CAT III"}.get(sev.lower(), sev or "?")
        body = (
            f"STIG-ID: {vid or rid}  Severity: {sev_label}  Title: {title}\n"
            f"Source: {source_name}\n\n"
            f"Discussion: {desc[:2000]}\n\n"
            f"Check Text: {check[:1500]}\n\n"
            f"Fix Text: {fix[:1500]}\n"
        )
        safe_id = re.sub(r"[^A-Za-z0-9_-]", "_", vid or rid)[:60]
        snippets.append((f"stig_{source_name}_{safe_id}.txt", body))
    return snippets

real_stig_count = 0
for zurl in REAL_STIG_ZIPS:
    src_name = re.sub(r"^U_|_STIG\.zip$", "", zurl.rsplit("/", 1)[-1])
    try:
        z = requests.get(zurl, timeout=120, headers={"User-Agent": "DISA-Workshop"})
        z.raise_for_status()
        with zipfile.ZipFile(io.BytesIO(z.content)) as zf:
            xccdf_names = [n for n in zf.namelist() if n.lower().endswith("-xccdf.xml")]
            if not xccdf_names:
                xccdf_names = [n for n in zf.namelist() if "Manual" in n and n.endswith(".xml")]
            for xn in xccdf_names[:1]:  # first XCCDF per zip is enough
                xb = zf.read(xn)
                snippets = xccdf_to_text(xb, src_name)
                # Write up to 50 snippets per source so we don't blow up the corpus
                for fname, body in snippets[:50]:
                    with open(f"{STIG_VOL}/{fname}", "w") as f:
                        f.write(body)
                print(f"  + {src_name}: wrote {min(len(snippets), 50)} STIG snippets")
                real_stig_count += min(len(snippets), 50)
    except Exception as e:
        print(f"  - {zurl}: {e}")

print(f"\nWrote {real_stig_count} real STIG control snippets")

STIGS = [
    ("stig_windows_server_2019_audit.txt", """STIG-ID: V-220701  Severity: CAT II  Title: Windows Server 2019 must enable auditing of privileged use.

Discussion: Privileged-use auditing tracks the use of administrator-level privileges. Without this, malicious privilege use cannot be detected.

Check Text: Verify the system is configured to audit "Privilege Use - Sensitive Privilege Use" with Success and Failure. Run AuditPol /get /category:* and confirm Sensitive Privilege Use is set to Success and Failure.

Fix Text: AuditPol /set /subcategory:"Sensitive Privilege Use" /success:enable /failure:enable

Related ATT&CK: T1078 (Valid Accounts), T1098 (Account Manipulation).
"""),
    ("stig_rhel8_ssh.txt", """STIG-ID: V-230229  Severity: CAT I  Title: RHEL 8 must not permit direct logons to the root account using SSH.

Discussion: Permitting direct root logon via SSH defeats accountability and increases credential-attack blast radius.

Check Text: grep -i permitrootlogin /etc/ssh/sshd_config; the value must be 'no'.

Fix Text: Edit /etc/ssh/sshd_config and set 'PermitRootLogin no'; restart sshd.

Related ATT&CK: T1021.004 (Remote Services: SSH).
"""),
    ("stig_cisco_ios_xe_http.txt", """STIG-ID: V-220976  Severity: CAT I  Title: Cisco IOS XE routers must disable the HTTP/HTTPS server unless explicitly required.

Discussion: The IOS XE Web UI has been the entry point for multiple critical CVEs (CVE-2023-20198) actively exploited in 2023 and 2024. Disabling the HTTP server eliminates this attack surface.

Check Text: show running-config | include http server. Both 'ip http server' and 'ip http secure-server' should be absent or replaced by 'no ip http server' / 'no ip http secure-server'.

Fix Text: configure terminal; no ip http server; no ip http secure-server; end; write memory.

Related ATT&CK: T1190 (Exploit Public-Facing Application).
"""),
    ("stig_exchange_owa.txt", """STIG-ID: V-228460  Severity: CAT II  Title: Microsoft Exchange Server must enable Extended Protection for Authentication on OWA.

Discussion: Extended Protection mitigates relay and man-in-the-middle attacks against the OWA endpoint.

Check Text: Get-ExchangeServer | Get-AuthRedirect; confirm ExtendedProtectionTokenChecking is set to Allow or Require for the OWA virtual directory.

Fix Text: Run Set-OwaVirtualDirectory -Identity "OWA (Default Web Site)" -ExtendedProtectionTokenChecking Allow.

Related ATT&CK: T1557 (Adversary-in-the-Middle).
"""),
]

if real_stig_count < 5:
    print(f"Only {real_stig_count} real STIG snippets; adding synthetic fallbacks.")
    for fn, body in STIGS:
        with open(f"{STIG_VOL}/{fn}", "w") as f:
            f.write(body)
        print(f"Wrote synthetic STIG excerpt: {fn}")
else:
    print(f"Have {real_stig_count} real STIG snippets; skipping synthetic.")


## Verify

In [ ]:
spark.sql(f"""
SELECT 'kev_catalog' AS tbl, COUNT(*) AS rows FROM {BASE}.kev_catalog
UNION ALL SELECT 'cves', COUNT(*) FROM {BASE}.cves
UNION ALL SELECT 'attack_techniques', COUNT(*) FROM {BASE}.attack_techniques
UNION ALL SELECT 'attack_groups', COUNT(*) FROM {BASE}.attack_groups
UNION ALL SELECT 'affected_assets', COUNT(*) FROM {BASE}.affected_assets;
""").display()


In [ ]:
import os
adv = os.listdir(VOL_ADVISORIES)
stig = os.listdir(VOL_STIGS)
print(f"Advisories volume: {len(adv)} files")
for a in adv: print(" -", a)
print(f"STIGs volume: {len(stig)} files")
for s in stig: print(" -", s)


In [ ]:
# Seed the per-user _workshop_config table; downstream notebooks read these.
for k, v in [
    ("catalog", CATALOG),
    ("schema", SCHEMA),
    ("llm_endpoint", LLM_ENDPOINT),
]:
    cfg_set(k, v)
spark.table(CONFIG_TABLE).display()
